# Prism Unified: Best of Everything

**What we know:**
- Spectral shape alone: doesn't beat baseline (sweep result)
- Directions alone: 12x speedup (sweep result)
- Marathon mod wheel: prevents overfitting (race v2 result)
- Sprint mod wheel: 3.8x fast convergence (race v1 result)

**What we haven't tested:**
Directions + mod wheel together, with the mod wheel pulling toward
the FULL Prism target (spectral shape + directions), not just spectral.

The unified approach: EigenTransfer init + continuous modulation.
Sweep alignment strength × mod decay on held-out test set.

In [ ]:
# Cell 1: Setup
import os, subprocess, shutil, numpy as np
os.chdir('/content')
if os.path.exists('/content/nanogpt-prism'):
    shutil.rmtree('/content/nanogpt-prism')
subprocess.run(['git', 'clone', 'https://github.com/timepointai/nanogpt-prism-shakespeare.git',
                '/content/nanogpt-prism'], check=True)
os.chdir('/content/nanogpt-prism')
subprocess.run(['pip', 'install', '-q', 'transformers', 'tiktoken', 'datasets'])
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
subprocess.run(['python', 'data/shakespeare_char/prepare.py'], capture_output=True)

# Contiguous partition
train_all = np.array(np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r'))
test_data = np.array(np.memmap('data/shakespeare_char/val.bin', dtype=np.uint16, mode='r'))
split = int(len(train_all) * 0.80)
for name, val in [('shakespeare_eval', test_data), ('shakespeare_teacher', train_all[split:])]:
    d = f'data/{name}'
    os.makedirs(d, exist_ok=True)
    train_all[:split].astype(np.uint16).tofile(os.path.join(d, 'train.bin'))
    val.astype(np.uint16).tofile(os.path.join(d, 'val.bin'))
    shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d, 'meta.pkl'))
print('Ready.')

In [ ]:
# Cell 2: Train teacher + extract (full directions)
import subprocess, os
os.chdir('/content/nanogpt-prism')

if os.path.exists('.prism_cache/teacher/directions.pt'):
    print('Cached.')
else:
    print('Training teacher (2000 steps)...')
    subprocess.run(['python', 'train.py', 'config/train_shakespeare_char.py',
        '--dataset=shakespeare_teacher', '--max_iters=2000',
        '--eval_interval=2000', '--eval_iters=50', '--log_interval=2000',
        '--out_dir=out-teacher-unified', '--always_save_checkpoint=True',
        '--compile=False', '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, timeout=600)
    print('Extracting...')
    subprocess.run(['python', 'prism_extract.py',
        '--ckpt', 'out-teacher-unified/ckpt.pt',
        '--out', '.prism_cache/teacher',
    ], capture_output=True, timeout=120)
    print('Done.')

In [ ]:
# Cell 3: Unified sweep — directions + mod wheel combinations
import subprocess, time, re, json, os
os.chdir('/content/nanogpt-prism')

STEPS = 5000
EVAL = 100
SPEC = '--prism_spectra=.prism_cache/teacher/spectra.json'
DIRS = '--prism_directions=.prism_cache/teacher/directions.pt'

configs = [
    # Baseline
    ('baseline', ['--prism_init=False']),
    
    # Directions only (no mod wheel) — the 12x result from sweep
    ('dirs_only', ['--prism_init=True', '--prism_align=0.75', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50']),
    
    # Directions + Sprint mod (fast fade)
    ('dirs_sprint', ['--prism_init=True', '--prism_align=0.75', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999']),
    
    # Directions + Marathon mod (slow fade) — the unified best?
    ('dirs_marathon', ['--prism_init=True', '--prism_align=0.75', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    
    # Sweep alignment strength with marathon mod
    ('dirs_a50_marathon', ['--prism_init=True', '--prism_align=0.5', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    ('dirs_a25_marathon', ['--prism_init=True', '--prism_align=0.25', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    
    # Higher alignment with marathon mod
    ('dirs_a90_marathon', ['--prism_init=True', '--prism_align=0.9', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    
    # Stronger mod with directions
    ('dirs_strong_mod', ['--prism_init=True', '--prism_align=0.75', SPEC, DIRS,
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.02', '--prism_mod_decay=0.9999']),
    
    # Default LR (1e-3) with directions + marathon
    ('dirs_marathon_hiLR', ['--prism_init=True', '--prism_align=0.75', SPEC, DIRS,
        '--learning_rate=1e-3', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
]

all_curves = {}
for name, extra in configs:
    log_path = f'/content/uni_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'\n  {name}...')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             '--dataset=shakespeare_eval',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=250',
             f'--out_dir=out-uni-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'    ERR: {result.stderr[-300:]}')
        print(f'    {time.time()-t0:.0f}s')
    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m: val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val:
        best = min(val.values())
        best_s = min(val, key=val.get)
        at_5k = val.get(5000, val.get(max(val.keys()), 0))
        print(f'  [{name}] best: {best:.4f} @{best_s}  @5000: {at_5k:.4f}')

with open('/content/uni_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 4: Results
import json
curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/uni_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*75)
print('  UNIFIED SWEEP: Directions + Mod Wheel')
print('='*75)
print(f'  Baseline best: {bb:.4f} at step {bs} (held-out test set)')

print(f'\n{"Config":>22s}  {"Hit@":>6s}  {"Speed":>6s}  {"Best":>8s}  {"@step":>6s}  {"@5000":>8s}  {"Overfit":>7s}')
print('-' * 72)
for name in sorted(curves.keys(), key=lambda n: min(curves[n].values()) if curves[n] else 999):
    c = curves[name]
    if not c: continue
    best = min(c.values())
    best_s = min(c, key=c.get)
    at_5k = c.get(5000, c.get(max(c.keys()), 0))
    hit = next((s for s in sorted(c) if c[s] <= bb), None)
    spd = f'{bs/hit:.1f}x' if hit else '—'
    hit_str = str(hit) if hit else 'never'
    # Check overfit: is val@5000 > best * 1.05?
    overfit = 'YES' if at_5k > best * 1.05 else 'no'
    print(f'{name:>22s}  {hit_str:>6s}  {spd:>6s}  {best:>8.4f}  {best_s:>6d}  {at_5k:>8.4f}  {overfit:>7s}')

# Key steps
print(f'\n--- Val loss at key steps ---')
names = list(curves.keys())
print(f'{"Step":>6s}', end='')
for n in names: print(f'  {n[:9]:>9s}', end='')
print()
for step in [0, 100, 300, 500, 1000, 2000, 3000, 5000]:
    vals = {n: curves[n].get(step) for n in names}
    active = [v for v in vals.values() if v]
    if not active: continue
    best = min(active)
    print(f'{step:>6d}', end='')
    for n in names:
        v = vals[n]
        if v:
            m = '*' if abs(v-best)<0.002 else ' '
            print(f'  {v:>8.4f}{m}', end='')
        else:
            print(f'  {"":>9s}', end='')
    print()

In [ ]:
# Cell 5: Download
from google.colab import files
files.download('/content/uni_curves.json')